# FieldVision bridge: the team's models in the agents

Validated default model (`eu_logistic_v1`, 42/44 on Beltsville zero-shot), county totals shifted to a realistic silage share, and field-level answers from FieldVision's own evidence. See `README.md` in this folder.

Cells 1-3 are free. Cells 5-8 call Nemotron (a few cents).

In [ ]:
# 1. Setup
import os, sys, json
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (fake FieldVision toolbox, no credits)
from fieldvision_agents.fieldvision_bridge.tests.run import run_all
run_all()
from fieldvision_agents.react import test_supervisor as ts
for name, fn in vars(ts).items():
    if name.startswith('test_'):
        fn()
print('supervisor tests pass')

In [ ]:
# 3. The bridge on the Beltsville county (needs pyarrow in FieldVision/.deps, see 12_real_tools)
from fieldvision_agents.fieldvision_bridge import FieldVisionToolbox
toolbox = FieldVisionToolbox()
print('default model:', toolbox.standard_model)
print('prior:', toolbox.prior('24033', 2022))
routes = toolbox.route_options('24033', 2022)
for name, opt in routes['options'].items():
    print(f"{name:20s}", opt.get('note', ''))

In [ ]:
# 4. Optional: classify and aggregate without the agent, to see adjusted vs unadjusted totals
toolbox.classify('24033', 2022, 'standard')
agg = toolbox.aggregate('24033', 2022)
{k: agg[k] for k in ['silage_acres', 'silage_acres_low', 'silage_acres_high', 'silage_acres_unadjusted', 'grain_acres', 'abstained_acres', 'prior_silage_share', 'prior_applied', 'county_coverage', 'adjustment']}

In [ ]:
# 5. The Nemotron supervisor on the bridge (Lightning)
from fieldvision_agents.react.supervisor import NemotronClient, run_county
result = run_county('24033', 2022, FieldVisionToolbox(), NemotronClient())
print(json.dumps({k: result[k] for k in ['run_id', 'status', 'policies', 'critic', 'summary', 'tokens']}, indent=1))

In [ ]:
# 6. Field-level questions, answered from FieldVision's evidence
from fieldvision_agents.fieldvision_bridge import ask_fields
fv = FieldVisionToolbox(); fv.classify('24033', 2022, result['policies'][-1] if result['policies'] else 'standard')
for q in ['Which fields abstained, and why?',
          'Which field is the model least sure about, and what does its satellite evidence show?',
          'Explain the largest silage-like field to a skeptic.']:
    a = ask_fields(fv, '24033', 2022, result, q)
    print('Q:', q, '\nA:', a['answer'], '\n   tools=', a['tools_used'], 'grounded=', a['grounded'], '\n')

In [ ]:
# 7. The report for this run (Ultra writes, Ultra edits, code checks numbers)
from IPython.display import Markdown, display
from fieldvision_agents.reporter.reporter import write_report
rep = write_report(result)
display(Markdown(rep['report_md'])); print(rep['grounding'])

In [ ]:
# 8. Planner on the real fields, with county-prior probabilities
import pandas as pd
from fieldvision_agents.planner.planner import county_status, plan_visits, supply_impact
frame = fv.planner_frame('24033', 2022)
status = county_status(frame)
plan = plan_visits(frame, k=3)
print(status)
print(f"3 visits: ±{plan['plus_minus_before']} → ±{plan['plus_minus_after']} acres")
display(pd.DataFrame(plan['fields']))
supply_impact(status)